In [2]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from diffusers import StableDiffusionPipeline
from diffusers import StableDiffusion3Pipeline
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from prdc import compute_prdc
import open_clip
import random


In [3]:
import torch
import os
from diffusers import StableDiffusion3Pipeline
from transformers import CLIPTextModelWithProjection, T5EncoderModel, CLIPTokenizer, T5TokenizerFast

from diffusers import StableDiffusion3Pipeline
import torch

pipe = StableDiffusion3Pipeline.from_pretrained("stabilityai/stable-diffusion-3.5-large", torch_dtype=torch.bfloat16)
pipe = pipe.to("cuda")

#pipe = StableDiffusionPipeline.from_pretrained(
#    "runwayml/stable-diffusion-v1-5",
#    torch_dtype=torch.float16,
#    safety_checker=None
#).to("cuda")

pipe.load_lora_weights(
    "lora_output/large_3",
    weight_name="last.safetensors"
)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()


Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

No LoRA keys associated to SD3Transformer2DModel found with the prefix='transformer'. This is safe to ignore if LoRA state dict didn't originally have any SD3Transformer2DModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to re

AttributeError: 'StableDiffusion3Pipeline' object has no attribute 'enable_vae_slicing'

In [ ]:
METADATA = Path("/home/nshevtsova/metadata_clean.csv")
IMG_DIR = Path("/home/nshevtsova/BCN_clean")

df = pd.read_csv(METADATA)


In [ ]:
def generate_images(pipe, cls, n):

    prompt = f"{PROMPT_MAP_NEW[cls]} high resolution dermoscopy image"
    negative = "clinical photo, ruler, text, watermark, low quality"

    images = []

    for _ in range(n):
        with torch.no_grad(), torch.cuda.amp.autocast():
            img = pipe(
                prompt,
                negative_prompt=negative,
                num_images_per_prompt=1,
                guidance_scale=7.0,
                num_inference_steps=40
            ).images[0]

        images.append(img)

    return images

In [ ]:
def load_real_images(df, image_dir, cls, n=5):

    df_cls = df[df["class"] == cls]

    if len(df_cls) == 0:
        return []

    df_cls = df_cls.sample(n=min(n, len(df_cls)), random_state=42)

    images = []

    for isic_id in df_cls["isic_id"]:
        img_path = Path(image_dir) / f"{isic_id}.jpg"

        if img_path.exists():
            images.append(Image.open(img_path).convert("RGB"))

    print(f"Loaded images → real: {len(images)}")

    return images

In [ ]:
fid = FrechetInceptionDistance(feature=2048).to("cpu")

def pil_to_uint8_tensor(img):
    # PIL → uint8 tensor [3,H,W]
    x = np.array(img, dtype=np.uint8)
    x = torch.from_numpy(x).permute(2, 0, 1)  # HWC → CHW
    return x

def compute_fid(real_imgs, fake_imgs):
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)

    for img in real_imgs:
        x = pil_to_uint8_tensor(img).unsqueeze(0).to(device)
        fid_metric.update(x, real=True)

    for img in fake_imgs:
        x = pil_to_uint8_tensor(img).unsqueeze(0).to(device)
        fid_metric.update(x, real=False)

    fid_value = fid_metric.compute().item()
    return fid_value

In [ ]:
import torchvision.models as models

device = torch.device("cpu")

inception = models.inception_v3(
    weights=models.Inception_V3_Weights.IMAGENET1K_V1,
    transform_input=False
).to("cpu").eval()

inception.fc = torch.nn.Identity()

def extract_features(images):
    feats = []
    preprocess = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor()
    ])

    with torch.no_grad():
        for img in images:
            x = preprocess(img).unsqueeze(0)
            f = inception(x)[0].detach().cpu().numpy()
            feats.append(f)

    return np.vstack(feats)


In [ ]:
def compute_prd(real_imgs, fake_imgs):
    real_feats = extract_features(real_imgs)
    fake_feats = extract_features(fake_imgs)

    return compute_prdc(
        real_features=real_feats,
        fake_features=fake_feats,
        nearest_k=5
    )


In [ ]:
device = torch.device("cpu")

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)
clip_model = clip_model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")


In [ ]:
def clip_similarity(images, text):
    text_tokens = tokenizer([text])

    sims = []
    with torch.no_grad():
        text_feat = clip_model.encode_text(text_tokens).float()
        text_feat /= text_feat.norm(dim=-1, keepdim=True)

        for img in images:
            img_tensor = clip_preprocess(img).unsqueeze(0)
            img_feat = clip_model.encode_image(img_tensor).float()
            img_feat /= img_feat.norm(dim=-1, keepdim=True)

            sims.append((img_feat @ text_feat.T).item())

    return np.mean(sims)


In [ ]:
MIN_REAL = 100
N_GEN = 100

results = []

DIAGNOSIS_MAP = {
    "Solar or actinic keratosis": "AK",
    "Basal cell carcinoma": "BCC",
    "Seborrheic keratosis": "BKL",
    "Solar lentigo": "BKL",
    "Dermatofibroma": "DF",
    "Melanoma metastasis": "MEL",
    "Melanoma, NOS": "MEL",
    "Nevus": "NV",
    "Squamous cell carcinoma, NOS": "SCC",
}

PROMPT_MAP = {
    "AK": "Actinic keratosis",
    "BCC": "Basal cell carcinoma",
    "BKL": "Benign keratosis lesion",
    "DF": "Dermatofibroma",
    "MEL": "Melanoma",
    "NV": "Nevus",
    "SCC": "Squamous cell carcinoma",
}

PROMPT_MAP_NEW = {
    "AK": "actinic keratosis skin lesion with rough scaly surface",
    "BCC": "basal cell carcinoma skin cancer with pearly border and visible blood vessels",
    "BKL": "benign keratosis skin lesion with waxy stuck-on appearance", 
    "DF": "dermatofibroma benign skin lesion with firm structure and central white scar-like area", 
    "MEL": "malignant melanoma skin cancer with asymmetric shape irregular border and multiple colors", 
    "NV": "benign melanocytic nevus mole with symmetric round shape and uniform brown pigmentation",
    "SCC": "squamous cell carcinoma skin cancer with scaly surface and irregular structure",
}

df = pd.read_csv(METADATA)
df["class"] = df["diagnosis_3"].map(DIAGNOSIS_MAP)

# удаляем все строки без класса
df = df.dropna(subset=["class"])
classes = sorted(df["class"].unique())

print(f"Found {len(classes)} classes: {classes}")

for cls in classes:

    print(f"\n=== Class {cls} ===")

    real_imgs = load_real_images(
        df=df,
        image_dir=IMG_DIR,
        cls=cls,
        n=N_GEN
    )

    if len(real_imgs) < 2:
        print(f"[SKIP] Not enough real images for {cls}")
        continue

    fake_imgs = generate_images(
        pipe,
        cls,
        n=N_GEN
    )

    fid = compute_fid(real_imgs, fake_imgs)

    prd = compute_prd(real_imgs, fake_imgs)

    clip_sim = clip_similarity(
        fake_imgs,
        f"dermoscopy image of {PROMPT_MAP[cls]}"
    )

    results.append({
        "class": cls,
        "n_real": len(real_imgs),
        "fid": fid,
        "precision": prd["precision"],
        "recall": prd["recall"],
        "clip": clip_sim,
    })

    print(f"FID: {fid:.2f}")
    print(f"Precision: {prd['precision']:.3f}")
    print(f"Recall: {prd['recall']:.3f}")
    print(f"CLIP: {clip_sim:.3f}")

    del fake_imgs
    torch.cuda.empty_cache()
